# EcoScan PK - Advanced: High-Accuracy Material Classifier Training Notebook
This notebook demonstrates how to load a pre-trained **MobileNetV2** model and perform **two-stage fine-tuning** on the **MINC-2500 / Construction Materials** dataset. This method adapts the model specifically to texture recognition and achieves high classification accuracy.

### Two-Stage Fine-Tuning Strategy:
1. **Stage 1 (Warmup Classification Head):** Freeze MobileNetV2 base layers and train the newly added dense layers for 10 epochs. This prevents random initial gradients from destroying pre-trained weights.
2. **Stage 2 (Fine-Tuning Base Weights):** Unfreeze the base model layers and train the entire network with a **very small learning rate (1e-5)** for another 15 epochs to specialize the feature detectors for construction materials.

## 1. Setup Environment

In [ ]:
import os
import numpy as np
import matplotlib.pyplot as plt
import tensorflow as tf
from tensorflow.keras import layers, models, applications

print("TensorFlow version:", tf.__version__)
print("GPU Available:", tf.config.list_physical_devices('GPU'))

## 2. Prepare Large Dataset
We increase the limit to **1500 images per category** to give the model plenty of training samples.

In [ ]:
CLASSES = ['Brick', 'Concrete', 'Glass', 'Steel', 'Wood', 'Marble', 'Granite', 'Tile', 'PVC', 'Paint']
DATASET_DIR = './dataset'

# Create directories for each class
for c in CLASSES:
    os.makedirs(os.path.join(DATASET_DIR, c), exist_ok=True)
print("Dataset directories initialized!")

In [ ]:
# Install HuggingFace datasets library
!pip install -q datasets

import os
from tqdm.notebook import tqdm
from datasets import load_dataset

print("📥 Downloading MINC-2500 split dataset from HuggingFace...")
ds = load_dataset("mcimpoi/minc-2500_split_1")

print("📂 Organizing images into category directories...")

# Increased limit to 1500 images per class for high-accuracy training
class_counts = {c: 0 for c in CLASSES}
limit_per_class = 1500

# Save images into local category folders for Keras training
for split in ['train', 'validation']:
    for idx, item in enumerate(tqdm(ds[split], desc=f"Processing {split} split")):
        label_idx = item['label']
        label_name = ds[split].features['label'].names[label_idx]
        
        # Map MINC labels to our folder structure
        folder_name = label_name.capitalize()
        if folder_name == 'Polishedstone': folder_name = 'Marble'
        if folder_name == 'Plastic': folder_name = 'PVC'
        if folder_name == 'Metal': folder_name = 'Steel'
        if folder_name == 'Painted': folder_name = 'Paint'
        
        # Only save if within target classes and class count limit
        if folder_name in CLASSES and class_counts[folder_name] < limit_per_class:
            save_dir = os.path.join(DATASET_DIR, folder_name)
            os.makedirs(save_dir, exist_ok=True)
            image_path = os.path.join(save_dir, f"{split}_{idx}.jpg")
            
            if not os.path.exists(image_path):
                item['image'].save(image_path)
                class_counts[folder_name] += 1

print("🎉 Dataset download and setup complete! Ready for training.")

## 3. Image Preprocessing & Strong Data Augmentation

In [ ]:
IMG_SIZE = (224, 224)
BATCH_SIZE = 32

# Strong augmentation to prevent overfitting on texture databases
data_augmentation = tf.keras.Sequential([
    layers.RandomFlip("horizontal_and_vertical"),
    layers.RandomRotation(0.25),
    layers.RandomZoom(0.15),
    layers.RandomContrast(0.15)
])

try:
    train_ds = tf.keras.utils.image_dataset_from_directory(
        DATASET_DIR,
        validation_split=0.2,
        subset="training",
        seed=123,
        image_size=IMG_SIZE,
        batch_size=BATCH_SIZE
    )
    
    val_ds = tf.keras.utils.image_dataset_from_directory(
        DATASET_DIR,
        validation_split=0.2,
        subset="validation",
        seed=123,
        image_size=IMG_SIZE,
        batch_size=BATCH_SIZE
    )
    
    # Prefetch datasets for GPU speed
    AUTOTUNE = tf.data.AUTOTUNE
    train_ds = train_ds.shuffle(1000).prefetch(buffer_size=AUTOTUNE)
    val_ds = val_ds.prefetch(buffer_size=AUTOTUNE)
    print("Datasets loaded successfully!")
except Exception as e:
    print("Error loading dataset:", e)

## 4. Build Model Structure (MobileNetV2)

In [ ]:
base_model = applications.MobileNetV2(
    input_shape=(224, 224, 3),
    include_top=False,
    weights='imagenet'
)
# Step 1: Lock the base model
base_model.trainable = False

model = models.Sequential([
    layers.Input(shape=(224, 224, 3)),
    data_augmentation,
    layers.Rescaling(1./255),
    base_model,
    layers.GlobalAveragePooling2D(),
    layers.Dropout(0.3),
    layers.Dense(128, activation='relu'),
    layers.Dense(len(CLASSES), activation='softmax')
])

model.compile(
    optimizer=tf.keras.optimizers.Adam(learning_rate=0.001),
    loss='sparse_categorical_crossentropy',
    metrics=['accuracy']
)
model.summary()

## 5. Training Phase 1: Classification Head Warmup (10 Epochs)

In [ ]:
history_warmup = model.fit(
    train_ds,
    validation_data=val_ds,
    epochs=10
)

## 6. Training Phase 2: Unfreeze & Deep Fine-Tuning (15 Epochs)
Here, we unfreeze the base layers and train the entire network with an extremely low learning rate (`1e-5`). This fine-tunes the MobileNet feature filters to classify construction textures.

In [ ]:
# Unfreeze base model
base_model.trainable = True

# Recompile model with a very low learning rate (1e-5)
model.compile(
    optimizer=tf.keras.optimizers.Adam(learning_rate=1e-5),
    loss='sparse_categorical_crossentropy',
    metrics=['accuracy']
)

# Train again for 15 epochs
history_finetune = model.fit(
    train_ds,
    validation_data=val_ds,
    epochs=15
)

## 7. Plot Accuracy & Loss Curves

In [ ]:
# Merge history stats
acc = history_warmup.history['accuracy'] + history_finetune.history['accuracy']
val_acc = history_warmup.history['val_accuracy'] + history_finetune.history['val_accuracy']
loss = history_warmup.history['loss'] + history_finetune.history['loss']
val_loss = history_warmup.history['val_loss'] + history_finetune.history['val_loss']

epochs_range = range(len(acc))

plt.figure(figsize=(14, 5))
plt.subplot(1, 2, 1)
plt.plot(epochs_range, acc, label='Training Accuracy')
plt.plot(epochs_range, val_acc, label='Validation Accuracy')
plt.axvline(x=9, color='r', linestyle='--', label='Fine-Tuning Start')
plt.legend(loc='lower right')
plt.title('Accuracy curves')

plt.subplot(1, 2, 2)
plt.plot(epochs_range, loss, label='Training Loss')
plt.plot(epochs_range, val_loss, label='Validation Loss')
plt.axvline(x=9, color='r', linestyle='--', label='Fine-Tuning Start')
plt.legend(loc='upper right')
plt.title('Loss curves')
plt.show()

## 8. Export Optimized Weights

In [ ]:
try:
    model.save('material_classifier.h5')
    print("✅ Saved high-accuracy weights file 'material_classifier.h5'!")
    from google.colab import files
    files.download('material_classifier.h5')
except Exception as e:
    print("Error exporting model:", e)